# Sampler Validation: Marginal Checks

For each validation target, we run the Boomerang and Sticky Boomerang,
resample uniformly in time, and overlay sample histograms against the
known marginal densities.

In [ ]:
import os
os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt

from benchmarks_august.targets.validation_sticky import spike_and_slab_gaussian
from benchmarks_august.samplers.factories import build_sampler
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.samplers.warmstart import warmup_reference

In [ ]:
# ── Shared settings ──────────────────────────────────────────────
N_SKELETON  = 20000
N_RESAMPLE  = 500000
BURNIN_FRAC = 0.1
refresh_rate = 1.0

def run_and_resample(sampler, target, sticky=False, warmup=True):
    """Warmup, preprocess, sample, and return time-uniform resamples."""
    if warmup:
        warmup_reference(sampler, n_rounds=3, n_pilot=500,
                         sticky=sticky, target=target)
    else:
        method = target.meta.get('preprocess_method', 'diagonal')
        if method == 'manual':
            sampler.preprocess(method='manual',
                               x_ref=target.x_ref,
                               Sigma_inv=target.Sigma_inv)
        else:
            sampler.preprocess(method='diagonal')
    
    sampler.reset(N=N_SKELETON)
    sampler.sample_auto(diagnostics=False)
    
    if sticky:
        _, samples = resample_sticky_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                               burnin_frac=BURNIN_FRAC)
    else:
        _, samples = resample_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                        burnin_frac=BURNIN_FRAC)
    return samples


def plot_marginals(target, samples_dict, figname=None):
    """Plot marginal histograms against true densities for each coordinate."""
    marginals = target.meta['marginal_grids']
    D = target.D
    n_samplers = len(samples_dict)
    
    fig, axes = plt.subplots(n_samplers, D, figsize=(3.5 * D, 3 * n_samplers),
                             squeeze=False)
    
    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']
    
    for row, (label, samples) in enumerate(samples_dict.items()):
        for col in range(D):
            ax = axes[row, col]
            mg = marginals[col]
            
            ax.hist(samples[:, col], bins=120, density=True, alpha=0.5,
                    color=colors[row % len(colors)], label=label)
            ax.plot(mg['grid'], mg['pdf'], 'k-', lw=1.5, label='True')
            
            if row == 0:
                ax.set_title(mg['label'])
            if col == 0:
                ax.set_ylabel(label)
            if row == n_samplers - 1:
                ax.set_xlabel(mg['label'])
            ax.legend(fontsize=7, loc='upper right')
    
    fig.suptitle(target.name, fontsize=14, y=1.02)
    fig.tight_layout()
    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight')
    plt.show()
    
    

## 0. Gaussian sanity check

In [ ]:
target_ss = spike_and_slab_gaussian(D=5, sigma=1.0,
                                     spike_weights=[0.8, 0.6, 0.4, 0.2, 0.1])
kappa_ss = target_ss.meta['kappa']

s_sticky = build_sampler('sticky_boomerang', target_ss, N=N_SKELETON, kappa=kappa_ss)
samp_sticky = run_and_resample(s_sticky, target_ss, sticky=True, warmup=False)

s_sticky_pli = build_sampler('sticky_boomerang_pli', target_ss, N=N_SKELETON, kappa=kappa_ss)
samp_sticky_pli = run_and_resample(s_sticky_pli, target_ss, sticky=True, warmup=False)


# Check: should be zero bounces
df = s_sticky.diagnostics_df
n_bounces = df[(df['event_type'] == 'bounce') & (df['accepted'] == True)].shape[0]
n_refresh = df[df['event_type'] == 'refresh'].shape[0]
wall = df['wall_seconds'].sum()
grad_evals = s_sticky.gradient_evals
df_pli = s_sticky_pli.diagnostics_df
n_bounces_pli = df_pli[(df_pli['event_type'] == 'bounce') & (df_pli['accepted'] == True)].shape[0]
n_refresh_pli = df_pli[df_pli['event_type'] == 'refresh'].shape[0]
wall_pli = df_pli['wall_seconds'].sum()
grad_evals_pli = s_sticky_pli.gradient_evals

print("--------- Boomerang ---------")
print(f"Accepted bounces: {n_bounces}  (expect 0)")
print(f"Refreshments:     {n_refresh}  (expect all skeleton points)")
print(f"Walltime:         {wall}")
print(f"Grad evals per skeleton point: {grad_evals / s_sticky.N:.1f}")
print("--------- Boomerang PLI ---------")
print(f"Accepted bounces: {n_bounces_pli}  (expect 0)")
print(f"Refreshments:     {n_refresh_pli}  (expect all skeleton points)")
print(f"Walltime:         {wall_pli}")
print(f"Grad evals per skeleton point: {grad_evals_pli / s_sticky_pli.N:.1f}")

plot_marginals(target_ss, {'Boomerang': samp_sticky,
                              'Boomerang PLI': samp_sticky_pli})
               #,figname='validation_gaussian_refcheck.pdf')